In [0]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql import functions as F

file_path = "/Volumes/workspace/default/health_data_volume/base_cdi.csv"

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

In [0]:
columns_to_drop = [
    "LocationAbbr", "DataSource", "DataValueAlt", "DataValueFootnote", "DataValueFootnoteSymbol", 
    "LowConfidenceLimit", "HighConfidenceLimit", "Geolocation", "LocationID", 
    "StratificationCategory2", "StratificationCategory3", "Stratification2", "Stratification3", 
    "TopicID", "QuestionID", "ResponseID", "DataValueTypeID", "StratificationCategoryID1", 
    "StratificationCategoryID2", "StratificationCategoryID3", "StratificationID1", 
    "StratificationID2", "StratificationID3", "Question", "Response"
]

df_cleaned = df.drop(*columns_to_drop)

In [0]:
df_cleaned = df_cleaned.withColumn("DataValue", regexp_replace(col("DataValue"), ",", "").cast("double"))
    
# Drop rows where critical column values are null
df_cleaned = df_cleaned.dropna(subset=["DataValue", "YearStart", "YearEnd", "Topic", "StratificationCategory1", "Stratification1", "DataValueUnit"])

# Drop duplicates
df_cleaned = df_cleaned.dropDuplicates()

# Ensure "DataValue" column is numeric
with_cast = df_cleaned.withColumn("num_val", F.expr("try_cast(`DataValue` as double)"))

non_numeric_count = with_cast.filter(
    F.col("DataValue").isNotNull() & F.col("num_val").isNull()
).count()

print(non_numeric_count)

# Save only data from 2016-2020
df_filtered = df_cleaned.filter(
    (F.col("YearStart") >= 2016) & (F.col("YearEnd") <= 2020)
)

0


In [0]:
# Check for consistency in critical column values
df_filtered.select("YearStart").distinct().show()

+---------+
|YearStart|
+---------+
|     2016|
|     2018|
|     2020|
|     2019|
+---------+



In [0]:
df_filtered.select("YearEnd").distinct().show()

+-------+
|YearEnd|
+-------+
|   2020|
|   2019|
+-------+



In [0]:
df.select("StratificationCategory1").distinct().show()

+-----------------------+
|StratificationCategory1|
+-----------------------+
|         Race/Ethnicity|
|                Overall|
|                    Sex|
|                    Age|
|                  Grade|
+-----------------------+



In [0]:
df.select("Topic").distinct().show()

+--------------------+
|               Topic|
+--------------------+
|              Cancer|
|       Health Status|
|            Diabetes|
|             Alcohol|
|         Oral Health|
| Non-Medical Factors|
|Cardiovascular Di...|
|       Mental Health|
|              Asthma|
|           Arthritis|
|          Disability|
|        Immunization|
|Chronic Kidney Di...|
|Chronic Obstructi...|
|Cognitive Health ...|
|Nutrition, Physic...|
|     Maternal Health|
|             Tobacco|
|               Sleep|
+--------------------+



In [0]:
df.select("DataValueUnit").distinct().show()

+-------------------+
|      DataValueUnit|
+-------------------+
|            gallons|
|             Number|
|                  %|
|  cases per 100,000|
|              Years|
|        per 100,000|
|    cases per 1,000|
|cases per 1,000,000|
+-------------------+



In [0]:
df.select("Stratification1").distinct().show()

+--------------------+
|     Stratification1|
+--------------------+
|            Hispanic|
|Native Hawaiian o...|
|                Male|
|            Age 0-44|
|            Grade 11|
|Multiracial, non-...|
| Asian, non-Hispanic|
|              Female|
|           Age 45-64|
|            Age >=65|
|            Grade 12|
|             Overall|
| Black, non-Hispanic|
|            Grade 10|
|American Indian o...|
|           Age 18-44|
|             Grade 9|
| White, non-Hispanic|
|Asian or Pacific ...|
|            Age 6-11|
+--------------------+
only showing top 20 rows


In [0]:
df2 = df_filtered.withColumn(
    "col_name",
    F.concat_ws("_", "Topic", "StratificationCategory1", "Stratification1", "DataValueUnit")
)

df2 = df2.withColumn("col_name", F.regexp_replace("col_name", r"[^A-Za-z0-9_]", "_"))

df_pivot = (
    df2
    .groupBy("YearStart", "YearEnd", "LocationDesc")
    .pivot("col_name")
    .agg(F.first("DataValue"))
)

df_pivot = df_pivot.filter(~F.col("LocationDesc").isin(["Puerto Rico", "Virgin Islands", "District of Columbia", "Guam", "United States"]))

In [0]:
# Drop territories
df_pivot.show()

+---------+-------+--------------+---------------------------+--------------------------------------+----------------------------+-----------------------+----------------------------+-----------------------+---------------------------------------+---------------------------+----------------------+--------------------------------------+------------------------+------------------------+------------------------+-----------------------+------------------------------+-------------------------+-----------------------------------------+-------------------------------+----------------------------------------------------------------------------+-----------------------------------------------------------------------+---------------------------------------------------------------------------------------+-------------------------------------------------+--------------------------------------------+------------------------------------------------------------+----------------------------------------

In [0]:
df_pivot.write.mode("overwrite").format("parquet").save("/Volumes/workspace/default/health_data_volume/cleaned_base_cdi")